In [ ]:
##Train the GPT2 model on NES-MDB MIDI dataset

In [ ]:
!pip install miditok==3.0.6 miditoolkit==0.1.16 pretty_midi==0.2.9 -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 44.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.9/158.9 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 68.7 MB/s eta 0:00:00


In [ ]:
# %% [markdown]
# # NES-MDB Chiptune Transformer — Minimal, Robust Notebook (2025)
# - Tokenize raw MIDIs directly (monophonic skyline) with **MIDILike** (no durations)
# - Build dataset with short-clip support and fallback seq_len
# - Train a small GPT-like model (Transformers)
# - Generate continuation + try to write MIDI
#
# Works with:
#   torch (CUDA if available), transformers 4.4x, datasets 2.2x,
#   miditok 3.0.6, miditoolkit 0.1.16, pretty_midi 0.2.9, numpy 2.x

# =========================
# 0) Compatibility & env check
# =========================
# %%
import sys, os, json, random, numpy as np
from importlib.metadata import version, PackageNotFoundError

# NumPy 2.x removed np.int / np.bool etc; some libs still reference them
if not hasattr(np, "int"):    np.int = int
if not hasattr(np, "bool"):   np.bool = bool
if not hasattr(np, "float"):  np.float = float
if not hasattr(np, "object"): np.object = object

def pkgver(name: str) -> str:
    try: return version(name)
    except PackageNotFoundError: return "not-found"

import torch
print("Python:", sys.version)
print("Torch:", torch.__version__, "| CUDA build:", torch.version.cuda, "| cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

import transformers, datasets
print("Transformers:", pkgver("transformers"))
print("Datasets:",    pkgver("datasets"))
print("miditok:",     pkgver("miditok"))
print("miditoolkit:", pkgver("miditoolkit"))
print("pretty_midi:", pkgver("pretty_midi"))
print("numpy:",       np.__version__)



/tmp/ipython-input-1391862299.py:23: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"): np.object = object


Python: 3.12.11 (main, Jun  4 2025, 08:56:18) [GCC 11.4.0]
Torch: 2.8.0+cu126 | CUDA build: 12.6 | cuda available: True
GPU: Tesla T4
Transformers: 4.57.0
Datasets: 4.0.0
miditok: 3.0.6
miditoolkit: 0.1.16
pretty_midi: 0.2.9
numpy: 2.0.2


In [ ]:
import tarfile
from pathlib import Path

tar_path = Path("data/nesmdb_midi.tar.gz")
if tar_path.exists():
    print("Extracting uploaded file...")
    with tarfile.open(tar_path, 'r:gz') as tar:
        tar.extractall('data/')
    tar_path.unlink()
    print("✓ Extracted to data/nesmdb_midi/")
else:
    print("✗ File not found. Please upload nesmdb_midi.tar.gz to data/ folder")

Extracting uploaded file...


/tmp/ipython-input-1531700273.py:8: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall('data/')


✓ Extracted to data/nesmdb_midi/


In [ ]:

# =========================
# 1) Paths & knobs
# =========================
# %%
from pathlib import Path

REPO      = Path.cwd()
DATA_DIR  = REPO / "data" / "nesmdb_midi"   # <--- put your raw MIDIs here
WORK      = REPO / "nes_transformer"

TOK_DIR   = WORK / "tokens"       # token JSONs { "ids": [...] }
RUN_DIR   = WORK / "hf_runs"      # HF checkpoints / logs
SAMPLES   = WORK / "samples"      # generated MIDIs
for p in [WORK, TOK_DIR, RUN_DIR, SAMPLES]:
    p.mkdir(parents=True, exist_ok=True)

print("DATA_DIR:", DATA_DIR.resolve(), "exists:", DATA_DIR.exists())

# --- tokenization speed/quality knobs ---
SUBSET_N   = 2000      # process only this many raw MIDIs now (None = all)  << adjust for time
SEED       = 42
LO_PITCH   = 48        # C3
HI_PITCH   = 96        # C7
MAX_TICKS  = None      # e.g. 20000 to crop early for speed; None = full file (more robust)
USE_THREADS= False     # start serial for stability; flip to True for speed once it works
MAX_WORKERS= max(4, os.cpu_count() or 4)



DATA_DIR: /content/data/nesmdb_midi exists: True


In [ ]:

# =========================
# 2) Tokenize directly from RAW (MIDILike, skyline to 1-voice)
# =========================
# %%
# Patch miditoolkit.Note with a .duration property (needed by miditok's converter under the hood)
import miditoolkit
try:
    from miditoolkit.midi.containers import Note as MTKNote
except Exception:
    from miditoolkit import Note as MTKNote
if not hasattr(MTKNote, "duration"):
    MTKNote.duration = property(lambda self: self.end - self.start)

from miditoolkit import MidiFile, Instrument, Note
from miditok import MIDILike, TokenizerConfig, TokSequence
from tqdm import tqdm

# Minimal tokenizer config (avoid durations/tempos/rets complexity)
tok_config = TokenizerConfig(
    beat_res={(0,0):4},     # 16th grid for positions
    use_chords=False,
    use_rests=False,
    use_tempos=False,
    use_time_signatures=False,
    use_programs=False
)
tokenizer = MIDILike(tok_config)
print("Tokenizer: MIDILike | vocab_size:", tokenizer.vocab_size)

# Skyline melody extractor (ticks): always monophonic
def skyline_ticks(notes, min_dur=1):
    if not notes: return []
    times = sorted({n.start for n in notes} | {n.end for n in notes})
    out = []
    cur_pitch, cur_start = None, None
    for t in times:
        active = [n for n in notes if n.start <= t < n.end]
        if active:
            pitch = max(active, key=lambda n: n.pitch).pitch
            if pitch != cur_pitch:
                if cur_pitch is not None and (t - cur_start) >= min_dur:
                    out.append(Note(velocity=90, pitch=cur_pitch, start=cur_start, end=t))
                cur_pitch, cur_start = pitch, t
        else:
            if cur_pitch is not None and (t - cur_start) >= min_dur:
                out.append(Note(velocity=90, pitch=cur_pitch, start=cur_start, end=t))
            cur_pitch, cur_start = None, None
    return out

# Gather raw files (+ optional subset)
raw_all = sorted([p for ext in ("*.mid","*.midi","*.MID","*.MIDI") for p in DATA_DIR.rglob(ext)])
print("Raw MIDIs found:", len(raw_all))
if SUBSET_N is not None and SUBSET_N < len(raw_all):
    random.seed(SEED)
    raw = sorted(random.sample(raw_all, SUBSET_N))
    print(f"Using subset: {len(raw)} / {len(raw_all)}")
else:
    raw = raw_all
    print("Using ALL raw files")

def tokenize_one(path: Path) -> Path | None:
    out = TOK_DIR / (path.stem + ".json")
    if out.exists():
        return out
    try:
        mf = MidiFile(str(path))
        # collect notes (non-drum, in range), optional crop for speed
        cand = []
        for inst in mf.instruments:
            if inst.is_drum or not inst.notes:
                continue
            notes = inst.notes
            if MAX_TICKS is not None:
                notes = [n for n in notes if n.start < MAX_TICKS]
            notes = [n for n in notes if LO_PITCH <= n.pitch <= HI_PITCH]
            cand.extend(notes)
        if len(cand) < 4:
            return None

        mono = skyline_ticks(cand, min_dur=1)
        if len(mono) < 4:
            return None

        # Build tiny 1-track MIDI in memory (keeps original grid)
        one = MidiFile(ticks_per_beat=mf.ticks_per_beat)
        one.tempo_changes = mf.tempo_changes
        one.time_signature_changes = mf.time_signature_changes
        inst = Instrument(program=80, is_drum=False, name="lead")
        inst.notes = mono
        one.instruments = [inst]

        # Tokenize (MIDILike)
        toks = tokenizer.tokenize(one) if hasattr(tokenizer, "tokenize") else tokenizer(one)
        ids  = toks.ids if hasattr(toks, "ids") else toks
        if not ids:
            return None

        out.write_text(json.dumps({"ids": ids}))
        return out
    except Exception:
        return None

token_files = []
if USE_THREADS:
    from concurrent.futures import ThreadPoolExecutor, as_completed
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = [ex.submit(tokenize_one, p) for p in raw]
        for f in tqdm(as_completed(futures), total=len(futures), desc="Tokenizing (threads)"):
            res = f.result()
            if res is not None:
                token_files.append(res)
else:
    for p in tqdm(raw, desc="Tokenizing (serial)"):
        res = tokenize_one(p)
        if res is not None:
            token_files.append(res)

print("Token files written:", len(token_files), "→", TOK_DIR)

# Simple peek
if token_files:
    sample_ids = json.loads(token_files[0].read_text())["ids"]
    print("Sample token length:", len(sample_ids))



Tokenizer: MIDILike | vocab_size: 338
Raw MIDIs found: 5278
Using subset: 2000 / 5278


Tokenizing (serial):   0%|          | 0/2000 [00:00<?, ?it/s]/tmp/ipython-input-2574420059.py:93: UserWarning: You are using a depreciated `miditoolkit.MidiFile` object. MidiTokis now (>v3.0.0) using symusic.Score as MIDI backend. Your file willbe converted on the fly, however please consider using symusic.
  toks = tokenizer.tokenize(one) if hasattr(tokenizer, "tokenize") else tokenizer(one)
Tokenizing (serial): 100%|██████████| 2000/2000 [02:03<00:00, 16.15it/s]

Token files written: 0 → /content/nes_transformer/tokens


In [ ]:

# =========================
# 3) Simple 16th-grid tokenizer (no MidiTok)
# =========================
from pathlib import Path
from tqdm import tqdm
from dataclasses import dataclass
import json, os, random, math

from miditoolkit import MidiFile, Instrument, Note

# ---- Paths (redefine if kernel was restarted) ----
REPO      = Path.cwd()
DATA_DIR  = REPO / "data" / "nesmdb_midi"   # raw MIDIs here
WORK      = REPO / "nes_transformer"
TOK_DIR   = WORK / "tokens_simple"
SAMPLES   = WORK / "samples"
for p in [WORK, TOK_DIR, SAMPLES]:
    p.mkdir(parents=True, exist_ok=True)

print("DATA_DIR:", DATA_DIR.resolve(), "| exists:", DATA_DIR.exists())

# ---- Token vocabulary: PAD=0, REST=1, HOLD=2, P0..P127=3..130 ----
PAD_ID  = 0
REST_ID = 1
HOLD_ID = 2
PITCH_BASE = 3  # P0 maps to 3, P127 maps to 130
VOCAB_SIZE = PITCH_BASE + 128

def pitch_to_id(p: int) -> int:
    p = max(0, min(127, int(p)))
    return PITCH_BASE + p

def id_to_pitch(i: int) -> int | None:
    if i >= PITCH_BASE and i < PITCH_BASE + 128:
        return i - PITCH_BASE
    return None  # REST/HOLD/PAD

# ---- Monophonic skyline over ticks ----
def skyline_ticks(notes, min_dur_ticks: int = 1):
    if not notes:
        return []
    times = sorted({n.start for n in notes} | {n.end for n in notes})
    out = []
    cur_pitch, cur_start = None, None
    for t in times:
        active = [n for n in notes if n.start <= t < n.end]
        if active:
            pitch = max(active, key=lambda n: n.pitch).pitch
            if pitch != cur_pitch:
                if cur_pitch is not None and (t - cur_start) >= min_dur_ticks:
                    out.append((cur_pitch, cur_start, t))
                cur_pitch, cur_start = pitch, t
        else:
            if cur_pitch is not None and (t - cur_start) >= min_dur_ticks:
                out.append((cur_pitch, cur_start, t))
            cur_pitch, cur_start = None, None
    return out

# ---- Quantize monophonic notes to a 16th grid ----
def quantize_to_grid_16th(mf: MidiFile, mono_notes, max_steps: int | None = None):
    tpq = max(1, int(mf.ticks_per_beat))
    ticks_per_step = max(1, tpq // 4)  # 16th = TPQ/4
    if not mono_notes:
        return []

    max_tick = 0
    for pitch, s, e in mono_notes:
        max_tick = max(max_tick, e)

    total_steps = math.ceil(max_tick / ticks_per_step)
    if max_steps is not None:
        total_steps = min(total_steps, max_steps)

    seq = [REST_ID] * max(1, total_steps)

    cur_idx = 0
    for pitch, s, e in mono_notes:
        start_idx = int(round(s / ticks_per_step))
        end_idx   = int(max(start_idx + 1, round(e / ticks_per_step)))
        if max_steps is not None:
            start_idx = min(start_idx, max_steps - 1)
            end_idx   = min(end_idx,   max_steps)

        # fill leading rest if any
        while cur_idx < start_idx and cur_idx < len(seq):
            seq[cur_idx] = REST_ID
            cur_idx += 1

        if start_idx < len(seq):
            seq[start_idx] = pitch_to_id(pitch)
            cur_idx = start_idx + 1

        # fill holds
        while cur_idx < end_idx and cur_idx < len(seq):
            seq[cur_idx] = HOLD_ID
            cur_idx += 1

    return seq

# ---- File → tokens pipeline ----
def midi_to_token_ids(path: Path, lo_pitch: int | None = None, hi_pitch: int | None = None,
                      crop_ticks: int | None = None, max_steps: int | None = 2048):
    try:
        mf = MidiFile(str(path))
    except Exception:
        return None

    # collect non-drum notes (optionally filter pitch / crop)
    notes = []
    for inst in mf.instruments:
        if inst.is_drum or not inst.notes:
            continue
        ns = inst.notes
        if crop_ticks is not None:
            ns = [n for n in ns if n.start < crop_ticks]
        if lo_pitch is not None and hi_pitch is not None:
            ns = [n for n in ns if lo_pitch <= n.pitch <= hi_pitch]
        notes.extend(ns)

    if not notes:
        return None

    mono = skyline_ticks(notes, min_dur_ticks=1)
    if not mono:
        return None

    ids = quantize_to_grid_16th(mf, mono, max_steps=max_steps)
    # keep only reasonably sized sequences
    return ids if len(ids) >= 16 else None

# ---- Batch tokenization ----
SUBSET_N   = 2000          # start small to confirm; bump to 1500–3000 when it works
SEED       = 42
LO_PITCH   = None         # None means keep all pitches (widest, safest)
HI_PITCH   = None
CROP_TICKS = None         # set e.g. 20000 to speed up
MAX_STEPS  = 1024         # cap per piece

raw_all = sorted([p for ext in ("*.mid","*.midi","*.MID","*.MIDI") for p in DATA_DIR.rglob(ext)])
print("Raw files:", len(raw_all))
random.seed(SEED)
raw = sorted(random.sample(raw_all, min(SUBSET_N, len(raw_all))))
print("Using subset:", len(raw))

written = 0
for p in tqdm(raw, desc="Tokenizing (simple)"):
    out = TOK_DIR / f"{p.stem}.json"
    if out.exists():
        written += 1
        continue
    ids = midi_to_token_ids(p, lo_pitch=LO_PITCH, hi_pitch=HI_PITCH,
                            crop_ticks=CROP_TICKS, max_steps=MAX_STEPS)
    if ids is None:
        continue
    out.write_text(json.dumps({"ids": ids}))
    written += 1

print("Token files written:", written, "→", TOK_DIR)
print("VOCAB_SIZE:", VOCAB_SIZE, "| PAD/REST/HOLD ids:", PAD_ID, REST_ID, HOLD_ID)





DATA_DIR: /content/data/nesmdb_midi | exists: True
Raw files: 5278
Using subset: 2000


Tokenizing (simple): 100%|██████████| 2000/2000 [03:00<00:00, 11.05it/s]

Token files written: 1856 → /content/nes_transformer/tokens_simple
VOCAB_SIZE: 131 | PAD/REST/HOLD ids: 0 1 2


In [ ]:

# =========================
# 4) Build dataset (keeps short clips; 512→256→128 fallback)
# =========================
from datasets import Dataset
import json, random

def build_ds(token_dir: Path, seq_len: int, keep_short_min: int = 8, step_frac: float = 0.5):
    files = sorted(token_dir.glob("*.json"))
    sequences = []
    step = max(1, int(seq_len * step_frac))
    for p in files:
        try:
            ids = json.loads(p.read_text())["ids"]
        except Exception:
            continue
        if not ids:
            continue
        if len(ids) <= seq_len:
            if len(ids) >= keep_short_min:
                sequences.append({"input_ids": ids, "labels": ids.copy()})
            continue
        # sliding window
        for i in range(0, len(ids) - seq_len + 1, step):
            seq = ids[i:i+seq_len]
            sequences.append({"input_ids": seq, "labels": seq.copy()})
    return sequences

SEQ_TRY = [512, 256, 128]
final_sequences, final_len = None, None
for L in SEQ_TRY:
    seqs = build_ds(TOK_DIR, seq_len=L, keep_short_min=8, step_frac=0.5)
    print(f"SEQ_LEN={L} -> sequences: {len(seqs)}")
    if seqs:
        final_sequences, final_len = seqs, L
        break

if final_sequences is None:
    raise RuntimeError("No sequences produced. Increase SUBSET_N, set CROP_TICKS=None, or lower keep_short_min to 4, then re-run.")

random.shuffle(final_sequences)
ds = Dataset.from_list(final_sequences).train_test_split(test_size=0.05, seed=42)
print(f"USING SEQ_LEN={final_len} | train={len(ds['train'])} | test={len(ds['test'])}")
ds




SEQ_LEN=512 -> sequences: 1961
USING SEQ_LEN=512 | train=1862 | test=99


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 1862
    })
    test: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 99
    })
})

In [ ]:

# =========================
# 5) Define & train model (no eval during training)
# =========================
import torch
from transformers import GPT2Config, AutoModelForCausalLM, Trainer, TrainingArguments

vocab_size = VOCAB_SIZE
gpt_cfg = GPT2Config(
    vocab_size=vocab_size,
    n_positions=max(1024, final_len * 2),
    n_embd=256,
    n_layer=4,
    n_head=8,
    n_inner=1024,
)
model = AutoModelForCausalLM.from_config(gpt_cfg)

def collate(batch):
    PAD = PAD_ID
    maxlen = max(len(x["input_ids"]) for x in batch)
    input_ids, labels, attn = [], [], []
    for x in batch:
        seq = x["input_ids"]
        pad = [PAD] * (maxlen - len(seq))
        inp = seq + pad
        lab = seq + pad
        for j in range(len(seq), maxlen):
            lab[j] = -100  # mask pads
        input_ids.append(inp)
        labels.append(lab)
        attn.append([1]*len(seq) + [0]*len(pad))
    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
        "attention_mask": torch.tensor(attn, dtype=torch.long),
    }

BATCH = 8
args = TrainingArguments(
    output_dir=str(WORK / "hf_runs_simple"),
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH,
    learning_rate=3e-4,
    warmup_steps=200,
    num_train_epochs=3,
    logging_steps=50,
    # IMPORTANT: turn OFF eval to avoid numpy conversion in the eval loop
    eval_strategy="no",          # (use this new arg name on 4.44+)
    save_strategy="steps",
    save_steps=1000,
    report_to=[],
    fp16=torch.cuda.is_available(),
    optim="adamw_torch",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds["train"],
    data_collator=collate,
    # NOTE: don't pass eval_dataset here
)

trainer.train()



`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
50,3.445700
100,2.559600
150,2.543200
200,2.412000
250,2.250400
300,2.268700
350,2.228500
400,2.192600
450,2.212900
500,2.122800


TrainOutput(global_step=699, training_loss=2.3380357035580963, metrics={'train_runtime': 28.7327, 'train_samples_per_second': 194.413, 'train_steps_per_second': 24.328, 'total_flos': 51489424318464.0, 'train_loss': 2.3380357035580963, 'epoch': 3.0})

In [ ]:

# =========================
# 6) Generate continuation and write MIDI
# =========================
def tokens_to_midi(ids, out_path: Path, bpm: int = 140):
    """Decode simple tokens back to a single-track MIDI."""
    tpq = 480
    ticks_per_step = tpq // 4  # 16th grid
    cur_idx = 0
    notes = []

    i = 0
    while i < len(ids):
        tid = ids[i]
        pitch = id_to_pitch(tid)
        if pitch is None:
            i += 1
            continue
        # start note
        start = i
        j = i + 1
        while j < len(ids) and ids[j] == HOLD_ID:
            j += 1
        end = j
        start_tick = start * ticks_per_step
        end_tick   = max(start_tick + ticks_per_step, end * ticks_per_step)
        notes.append((pitch, start_tick, end_tick))
        i = j

    # write MIDI
    mf = MidiFile(ticks_per_beat=tpq)
    inst = Instrument(program=80, is_drum=False, name="lead")
    inst.notes = [Note(velocity=90, pitch=p, start=s, end=e) for (p, s, e) in notes]
    mf.instruments = [inst]
    # Add a constant tempo (approximate)
    from miditoolkit.midi.containers import TempoChange
    mf.tempo_changes = [TempoChange(tempo=bpm, time=0)]
    mf.dump(str(out_path))

# Pick a short seed (or any)
files = sorted(TOK_DIR.glob("*.json"))
assert files, "No token files found—rerun Step 3."
seed_path = min(files, key=lambda p: len(json.loads(p.read_text())["ids"]))
seed_ids  = json.loads(seed_path.read_text())["ids"]

inp = torch.tensor(seed_ids, dtype=torch.long)[None, :]
if torch.cuda.is_available():
    model.to("cuda"); inp = inp.to("cuda")

model.eval()
with torch.no_grad():
    gen = model.generate(
        input_ids=inp,
        max_new_tokens=256,
        do_sample=True,
        temperature=1.0,
        top_p=0.95,
        pad_token_id=PAD_ID
    )

out_ids = gen[0].tolist()
out_mid = SAMPLES / "sample_simple.mid"
tokens_to_midi(out_ids, out_mid, bpm=140)
print("Wrote:", out_mid)


Wrote: /content/nes_transformer/samples/sample_simple.mid


In [ ]:
# Load trained model and setup
import torch
import json
from pathlib import Path
from transformers import AutoModelForCausalLM

WORK = Path.cwd() / "nes_transformer"
checkpoint_dir = WORK / "hf_runs_simple"

# Find latest checkpoint
checkpoints = sorted([d for d in checkpoint_dir.glob("checkpoint-*")],
                     key=lambda x: int(x.name.split("-")[1]))
if checkpoints:
    latest_checkpoint = checkpoints[-1]
    print(f"Loading from: {latest_checkpoint}")
else:
    latest_checkpoint = checkpoint_dir
    print(f"Loading from: {latest_checkpoint}")

model = AutoModelForCausalLM.from_pretrained(latest_checkpoint)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()
print(f"✓ Model loaded on {device}")

Loading from: /content/nes_transformer/hf_runs_simple/checkpoint-699
✓ Model loaded on cuda


In [ ]:
# Helper functions for prediction
def notes_to_tokens(pitches, durations=None):
    """Convert pitches to tokens. Use None for rests."""
    if durations is None:
        durations = [1] * len(pitches)

    tokens = []
    for pitch, dur in zip(pitches, durations):
        if pitch is None:
            tokens.extend([REST_ID] * dur)
        else:
            tokens.append(pitch_to_id(pitch))
            if dur > 1:
                tokens.extend([HOLD_ID] * (dur - 1))
    return tokens

def tokens_to_notes(tokens):
    """Convert tokens back to (pitch, start, duration)."""
    notes = []
    i = 0
    while i < len(tokens):
        tid = tokens[i]
        pitch = id_to_pitch(tid)

        if pitch is None:
            i += 1
            continue

        start = i
        j = i + 1
        while j < len(tokens) and tokens[j] == HOLD_ID:
            j += 1

        duration = j - start
        notes.append((pitch, start, duration))
        i = j

    return notes

def predict_next_notes(input_pitches, num_tokens=32, temperature=1.0, top_p=0.95):
    """Predict next notes given input pitches."""
    input_tokens = notes_to_tokens(input_pitches)
    input_tensor = torch.tensor(input_tokens, dtype=torch.long).unsqueeze(0).to(device)

    print(f"Input: {input_pitches}")
    print(f"Token length: {len(input_tokens)}")

    with torch.no_grad():
        output = model.generate(
            input_ids=input_tensor,
            max_new_tokens=num_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            pad_token_id=PAD_ID
        )

    generated_tokens = output[0, len(input_tokens):].tolist()
    return generated_tokens

print("✓ Helper functions loaded")

✓ Helper functions loaded


In [ ]:
# Example: C major scale
input_notes = [60, 62, 64, 65, 67]  # C D E F G
generated = predict_next_notes(input_notes, num_tokens=32, temperature=0.9)

print(f"\nGenerated tokens: {generated[:20]}")
all_notes = tokens_to_notes(notes_to_tokens(input_notes) + generated)
print(f"\nFirst 10 notes (pitch, start, duration):")
for note in all_notes[:10]:
    print(f"  Pitch {note[0]:3d}, Start {note[1]:3d}, Duration {note[2]}")

Input: [60, 62, 64, 65, 67]
Token length: 5

Generated tokens: [70, 69, 70, 70, 70, 65, 70, 70, 70, 69, 70, 72, 70, 68, 68, 68, 70, 70, 74, 70]

First 10 notes (pitch, start, duration):
  Pitch  60, Start   0, Duration 1
  Pitch  62, Start   1, Duration 1
  Pitch  64, Start   2, Duration 1
  Pitch  65, Start   3, Duration 1
  Pitch  67, Start   4, Duration 1
  Pitch  67, Start   5, Duration 1
  Pitch  66, Start   6, Duration 1
  Pitch  67, Start   7, Duration 1
  Pitch  67, Start   8, Duration 1
  Pitch  67, Start   9, Duration 1


In [ ]:
# Generate and save as MIDI
input_notes = [60, 64, 67, 72]  # C E G C (octave)
generated = predict_next_notes(input_notes, num_tokens=128, temperature=0.9)
full_sequence = notes_to_tokens(input_notes) + generated

SAMPLES = WORK / "samples"
out_file = SAMPLES / "my_prediction.mid"
tokens_to_midi(full_sequence, out_file, bpm=140)

print(f"\n✓ Saved to: {out_file}")

# Download the file
from google.colab import files
files.download(str(out_file))

Input: [60, 64, 67, 72]
Token length: 4

✓ Saved to: /content/nes_transformer/samples/my_prediction.mid


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Test multiple starting patterns
test_patterns = [
    ([60, 62, 64, 65], "C major scale"),
    ([60, 60, 67, 67], "Twinkle twinkle"),
    ([69, 67, 65, 64], "Descending"),
    ([60, 64, 67], "C major chord"),
]

for pitches, name in test_patterns:
    print(f"\n=== {name} ===")
    print(f"Input: {pitches}")
    generated = predict_next_notes(pitches, num_tokens=32, temperature=0.9)
    notes = tokens_to_notes(generated[:20])
    print(f"Next notes: {[n[0] for n in notes]}")


=== C major scale ===
Input: [60, 62, 64, 65]
Input: [60, 62, 64, 65]
Token length: 4
Next notes: [60, 60, 65, 64, 60, 67]

=== Twinkle twinkle ===
Input: [60, 60, 67, 67]
Input: [60, 60, 67, 67]
Token length: 4
Next notes: [67, 67, 67, 72, 67, 60, 67, 67, 67]

=== Descending ===
Input: [69, 67, 65, 64]
Input: [69, 67, 65, 64]
Token length: 4
Next notes: [65, 67, 71, 67, 71, 67, 67, 69, 69, 67, 67, 64, 67, 72, 72, 66, 65, 65, 69]

=== C major chord ===
Input: [60, 64, 67]
Input: [60, 64, 67]
Token length: 3
Next notes: [62, 60, 71, 62, 62, 64, 57, 62, 67, 67, 67, 62, 55, 62, 60, 62, 62, 64, 62]


In [ ]:
# Define musical scales
SCALES = {
    'C_major': [0, 2, 4, 5, 7, 9, 11],  # C D E F G A B
    'C_minor': [0, 2, 3, 5, 7, 8, 10],  # C D Eb F G Ab Bb
    'C_pentatonic': [0, 2, 4, 7, 9],    # C D E G A
    'C_blues': [0, 3, 5, 6, 7, 10],     # C Eb F Gb G Bb
    'D_major': [2, 4, 6, 7, 9, 11, 1],  # D E F# G A B C#
    'G_major': [7, 9, 11, 0, 2, 4, 6],  # G A B C D E F#
    'A_minor': [9, 11, 0, 2, 4, 5, 7],  # A B C D E F G
}

def is_in_scale(pitch, scale_notes):
    """Check if a pitch is in the given scale."""
    if pitch is None:
        return True  # Rests are always allowed
    return (pitch % 12) in scale_notes

def quantize_to_scale(pitch, scale_notes):
    """Snap a pitch to the nearest note in the scale."""
    if pitch is None:
        return None

    pitch_class = pitch % 12
    if pitch_class in scale_notes:
        return pitch

    # Find nearest scale note
    octave = pitch // 12
    distances = [(abs(pitch_class - note), note) for note in scale_notes]

    # Handle wraparound (e.g., B to C)
    distances.extend([(abs(pitch_class - (note + 12)), note) for note in scale_notes])
    distances.extend([(abs(pitch_class - (note - 12)), note) for note in scale_notes])

    _, nearest = min(distances)
    nearest = nearest % 12  # Normalize to 0-11

    return octave * 12 + nearest

def filter_tokens_by_scale(tokens, scale_notes):
    """Filter generated tokens to only include scale notes."""
    filtered = []
    for tid in tokens:
        pitch = id_to_pitch(tid)

        if pitch is None:
            # Keep REST and HOLD tokens
            filtered.append(tid)
        elif is_in_scale(pitch, scale_notes):
            # Keep notes in scale
            filtered.append(tid)
        else:
            # Quantize out-of-scale notes
            new_pitch = quantize_to_scale(pitch, scale_notes)
            filtered.append(pitch_to_id(new_pitch))

    return filtered

print("✓ Scale constraint functions loaded")
print(f"Available scales: {list(SCALES.keys())}")

✓ Scale constraint functions loaded
Available scales: ['C_major', 'C_minor', 'C_pentatonic', 'C_blues', 'D_major', 'G_major', 'A_minor']


In [ ]:
def predict_with_scale_constraint_fixed(input_pitches, scale_name='C_major',
                                       num_tokens=64, temperature=1.0,
                                       penalty_strength=10.0):
    """
    Generate with scale constraint using logit biasing.
    Instead of hard masking, we penalize out-of-scale notes.
    """

    if scale_name not in SCALES:
        print(f"Warning: '{scale_name}' not found. Using C_major as fallback.")
        scale_name = 'C_major'

    scale_notes = SCALES[scale_name]
    print(f"Constraining to: {scale_name} → {scale_notes}")

    # Create penalty vector (not a mask)
    penalties = torch.zeros(VOCAB_SIZE, dtype=torch.float32, device=device)

    # Don't penalize special tokens
    penalties[PAD_ID] = 0
    penalties[REST_ID] = 0
    penalties[HOLD_ID] = 0

    # Penalize out-of-scale notes
    for pitch in range(128):
        token_id = pitch_to_id(pitch)
        if not is_in_scale(pitch, scale_notes):
            penalties[token_id] = penalty_strength

    # Use standard generation with logits processor
    input_tokens = notes_to_tokens(input_pitches)
    input_tensor = torch.tensor(input_tokens, dtype=torch.long).unsqueeze(0).to(device)

    # Define custom logits processor
    from transformers import LogitsProcessor, LogitsProcessorList

    class ScaleLogitsProcessor(LogitsProcessor):
        def __init__(self, penalties):
            self.penalties = penalties

        def __call__(self, input_ids, scores):
            # Subtract penalty from logits (makes out-of-scale notes less likely)
            scores = scores - self.penalties
            return scores

    logits_processor = LogitsProcessorList([
        ScaleLogitsProcessor(penalties)
    ])

    with torch.no_grad():
        output = model.generate(
            input_ids=input_tensor,
            max_new_tokens=num_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=0.95,
            pad_token_id=PAD_ID,
            logits_processor=logits_processor,
            repetition_penalty=1.2
        )

    generated_tokens = output[0, len(input_tokens):].tolist()
    return generated_tokens

# Test with C MINOR!
print("\n=== C MINOR Scale Constraint ===")
print("C minor notes: C D Eb F G Ab Bb")
print("MIDI pitches: 60 62 63 65 67 68 70\n")

input_notes = [60, 63, 65, 65, 67, 63]  # C Eb F G (C minor chord progression)
generated = predict_with_scale_constraint_fixed(
    input_notes,
    scale_name='C_minor',  # ✅ C MINOR!
    num_tokens=128,
    temperature=0.9,
    penalty_strength=15.0
)

# Show notes
all_tokens = notes_to_tokens(input_notes) + generated
notes = tokens_to_notes(all_tokens)
print(f"\nTotal notes generated: {len(notes)}")
print(f"First 20 pitches: {[n[0] for n in notes[:20]]}")

# Check if in C MINOR scale
scale_notes = SCALES['C_minor']  # [0, 2, 3, 5, 7, 8, 10]
pitches_only = [n[0] for n in notes]
out_of_scale = [p for p in pitches_only if not is_in_scale(p, scale_notes)]

print(f"\n📊 C Minor Scale Compliance:")
print(f"  Total pitches: {len(pitches_only)}")
print(f"  Out-of-scale: {len(out_of_scale)} ({len(out_of_scale)/len(pitches_only)*100:.1f}%)")
if out_of_scale:
    print(f"  Violations: {out_of_scale[:10]}")

# Save as MIDI
full_sequence = notes_to_tokens(input_notes) + generated
out_file = SAMPLES / "c_minor_melody.mid"
tokens_to_midi(full_sequence, out_file, bpm=30)
print(f"\n✅ Saved C MINOR melody to: {out_file}")

from google.colab import files
files.download(str(out_file))


=== C MINOR Scale Constraint ===
C minor notes: C D Eb F G Ab Bb
MIDI pitches: 60 62 63 65 67 68 70

Constraining to: C_minor → [0, 2, 3, 5, 7, 8, 10]

Total notes generated: 103
First 20 pitches: [60, 63, 65, 65, 67, 63, 62, 58, 58, 58, 67, 70, 70, 70, 70, 72, 72, 58, 48, 48]

📊 C Minor Scale Compliance:
  Total pitches: 103
  Out-of-scale: 0 (0.0%)

✅ Saved C MINOR melody to: /content/nes_transformer/samples/c_minor_melody.mid


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>